In [4]:
import pandas as pd

# =========================
# PARAMÈTRES À ADAPTER
# =========================
FICHIER = r"C:\Users\9920510N\SNCF\Datalab' Mass Transit GrpO365 - Documents\Rayonnement\Datathon\2026\Hackaviz IdFM\Données\Flux_DT_transport.csv"                # chemin du fichier CSV
SORTIE = r"C:\Users\9920510N\SNCF\Datalab' Mass Transit GrpO365 - Documents\Rayonnement\Datathon\2026\Hackaviz IdFM\Données\Resultats_DT_transport.csv"   # fichier de résultat

# =========================
# 1. OUVERTURE DU FICHIER
# =========================
def lire_csv(chemin):
    """Détecte automatiquement le séparateur (; , tabulation) et l'encodage."""
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(chemin, sep=None, engine="python",
                               dtype={"geocode_commune": str}, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError("Impossible de lire le fichier : encodage non reconnu.")

df = lire_csv(FICHIER)
df.columns = df.columns.str.strip()

# Suppression de la colonne date_mesure
df = df.drop(columns=["date_mesure"], errors="ignore")

# Nettoyage
df["geocode_commune"] = df["geocode_commune"].str.strip().str.zfill(5)   # garde les 0 initiaux et 2A/2B
df["mode_transport"] = df["mode_transport"].str.strip()
df["valeur"] = pd.to_numeric(
    df["valeur"].astype(str)
      .str.replace("\u00a0", "", regex=False)   # espaces insécables
      .str.replace(" ", "", regex=False)
      .str.replace(",", ".", regex=False),      # virgule décimale
    errors="coerce"
)

print(f"{len(df)} lignes lues | {df['geocode_commune'].nunique()} communes | "
      f"{df['mode_transport'].nunique()} modes de transport")
print(f"Valeurs non numériques ou manquantes : {df['valeur'].isna().sum()}")

# =========================
# 2. % PAR MODE DE TRANSPORT ET PAR COMMUNE
# =========================
# Somme des valeurs par commune et par mode
agg = df.groupby(["geocode_commune", "mode_transport"], as_index=False)["valeur"].sum()

# Part de chaque mode dans le total de la commune
agg["total_commune"] = agg.groupby("geocode_commune")["valeur"].transform("sum")
agg["pourcentage"] = (agg["valeur"] / agg["total_commune"] * 100).round(2)

# Ajout du libellé de la commune
libelles = df.groupby("geocode_commune")["libelle_commune"].first()
agg.insert(1, "libelle_commune", agg["geocode_commune"].map(libelles))

# Format large : une ligne par commune, une colonne par mode
resultat = (agg.pivot_table(index=["geocode_commune", "libelle_commune"],
                            columns="mode_transport",
                            values="pourcentage",
                            fill_value=0)
               .add_prefix("%_")
               .reset_index())
resultat.columns.name = None

# Vérification : chaque ligne doit totaliser environ 100 %
cols_pct = [c for c in resultat.columns if c.startswith("%_")]
resultat["total_%"] = resultat[cols_pct].sum(axis=1).round(1)
anomalies = resultat[(resultat["total_%"] - 100).abs() > 0.5]
print(f"Communes dont le total diffère de 100 % (total nul ou valeurs manquantes) : {len(anomalies)}")

#Duplication des résultats pour Paris à chaque arrondissemnt
codes = [f"751{i:02d}" for i in range(1, 21)]

def dupliquer_paris(df):
    df = df.copy()
    df["geocode_commune"] = df["geocode_commune"].astype(str).str.zfill(5)
    paris = df[df["geocode_commune"] == "75056"]

    copies = pd.concat([
        paris.assign(
            geocode_commune=code,
            libelle_commune=f"Paris {i}{'er' if i == 1 else 'e'} Arrondissement"
        )
        for i, code in enumerate(codes, start=1)
    ], ignore_index=True)

    return pd.concat(
        [df[df["geocode_commune"] != "75056"], copies],
        ignore_index=True
    )

resultat = dupliquer_paris(resultat)
agg = dupliquer_paris(agg)
# =========================
# EXPORT (format compatible Excel français)
# =========================
SORTIE_LONG = SORTIE.replace(".csv", "_format_long.csv")
resultat.to_csv(SORTIE, sep=";", decimal=",", index=False, encoding="utf-8-sig")
#agg.to_csv(SORTIE_LONG, sep=";", decimal=",", index=False, encoding="utf-8-sig")

print(resultat.head(10))
#print(f"\nFichiers créés : {SORTIE} et {SORTIE_LONG}")

125944 lignes lues | 34758 communes | 6 modes de transport
Valeurs non numériques ou manquantes : 0
Communes dont le total diffère de 100 % (total nul ou valeurs manquantes) : 0
  geocode_commune          libelle_commune  %_Deux-roues motorisé  %_Marche  \
0           01001  L'Abergement-Clémenciat                   2.22      0.00   
1           01002    L'Abergement-de-Varey                   0.00      0.00   
2           01004        Ambérieu-en-Bugey                   0.85      7.86   
3           01005      Ambérieux-en-Dombes                   0.56      1.74   
4           01006                  Ambléon                   0.00      0.00   
5           01007                 Ambronay                   1.13      1.14   
6           01008                 Ambutrix                   1.38      2.77   
7           01009         Andert-et-Condon                   0.00      3.21   
8           01010                Anglefort                   1.05      0.00   
9           01011               